# Class 6 — FDE Data Validation with FlashEats

Yesterday, you proved you could **retrieve** data from multiple systems.

Today the question is different:

> **Can we safely use that data to make a business decision?**

This is not a generic data-cleaning lab.

## FDE validation mindset

**Business claim → assumptions → validation contract → targeted checks → stakeholder clarification → PASS / WARN / FAIL → publish decision**

### Concepts

**1. Validation is decision-dependent**  
A field can be good enough for weekly reporting but unsafe for a live operational decision.

**2. Turn assumptions into contracts**  
Business assumption → data expectation → executable check → action.

**3. Separate three validation types**
- Technical: chronology, IDs, categories, mappings
- Semantic: what does a status or ETA actually mean?
- Organizational: who owns the KPI definition?

**4. Never silently fix ambiguity**  
Do not encode thresholds or category mappings without ownership.

**5. Output a validation gate**  
The deliverable is not merely a cleaned dataframe. It is a decision:
PASS / WARN / FAIL / UNKNOWN.

In [1]:
# [replaced classroom cell] The Colab/zip loader and `!pip install` are not needed here:
# the client pack lives in source_systems/ and dependencies come from requirements.txt.
import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "source_systems").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
BASE = ROOT / "source_systems"                    # read-only client systems
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "Challenges" / "output"             # scratch outputs of the challenge notebooks
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 90)
print("BASE:", BASE, "| database exists:", (BASE / "database" / "flasheats.db").exists())

BASE: B:\Trimester-9\FDE\assignment-2\source_systems | database exists: True


## Load the updated FlashEats data

In [2]:
con = sqlite3.connect(BASE / "database" / "flasheats.db")

orders = pd.read_sql("SELECT * FROM orders", con)
restaurants = pd.read_sql("SELECT * FROM restaurants", con)
drivers = pd.read_sql("SELECT * FROM drivers", con)

tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")

with open(BASE / "data" / "client_metric_definitions.json", "r") as f:
    metric_notes = json.load(f)

print("orders:", orders.shape)
print("tickets:", tickets.shape)
print("restaurant_status:", restaurant_status.shape)

display(orders.head())
display(tickets.head())
display(restaurant_status.head())
print(json.dumps(metric_notes, indent=2))

orders: (1603, 13)
tickets: (202, 5)
restaurant_status: (502, 4)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear
2,O00003,C0855,R010,D039,Bengaluru,2026-08-01T19:35:00,2026-08-01T20:17:52.941967,2026-08-01T19:57:50.820366,2026-08-01T20:13:28.061191,delivered,2.42,medium,clear
3,O00004,C0229,R030,D038,Bengaluru,2026-08-09T18:16:00,2026-08-09T19:30:19.067006,2026-08-09T18:46:48.725485,NaN,cancelled,14.20,high,rain
4,O00005,C0040,R004,D047,Bengaluru,2026-08-06T20:35:00,2026-08-06T21:54:48,2026-08-06T21:18:38.024280,2026-08-06T22:15:49.290713,delivered,18.00,high,clear


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.
2,T00003,NaN,2026-08-25T18:22:00,status_mismatch,The app says picking up but the restaurant says it is ready.
3,T00004,O00037,2026-08-09T21:09:00,Late Delivery,The rider has not moved for 15 minutes.
4,T00005,O00040,2026-08-11T20:15:00,late_delivery,My order is already past the promised time.


,order_id,restaurant_id,status,last_updated_at
0,O00676,R040,READY,2026-08-17T13:29:00
1,O01506,R055,Ready,2026-08-27T16:37:00
2,O01484,R032,ready,2026-08-08T19:44:00
3,O00675,R012,handoff,2026-08-22T21:42:00
4,O00078,R027,unknown,2026-08-25T19:35:00


{
  "metric_under_review": "Late Delivery Rate",
  "leadership_claim": "Late Delivery Rate is 56%",
  "stakeholders": {
    "VP Operations": "Any delivered order after the promised ETA is late.",
    "Support Lead": "Only more than 10 minutes beyond ETA should count as meaningfully late.",
    "Finance": "Cancelled/refunded orders should not count in operational performance.",
    "Data Team": "Historical dashboard uses delivered orders with non-null actual delivery time."
  },
  "note": "No canonical KPI owner is formally documented."
}


# Challenge 1 — Can we defend the “56% late” claim?


Leadership says:

> **“Late Delivery Rate is 56%.”**

Before calculating anything, create a validation contract.

| Business assumption | Data expectation | How will you test it? | Severity if false |
|---|---|---|---|
| One row = one business order |  |  |  |
| Delivered orders have completion time |  |  |  |
| Promised ETA is valid |  |  |  |
| Event chronology is valid |  |  |  |
| “Late” has an agreed definition |  |  |  |

**Hint:** Don't start by cleaning. Ask: *what could make 56% misleading?*

> ✅ **Answer**

| Business assumption | Data expectation | How it is tested | Severity if false |
|---|---|---|---|
| One row = one business order | `order_id` unique | rows vs `nunique()`; do the duplicates disagree? | HIGH: the denominator inflates |
| Delivered orders have a completion time | delivered ⇒ `actual_delivery_at` not null | count nulls among delivered orders | HIGH: the population shrinks with unknown bias |
| Promised ETA is valid | parseable and `promised_eta ≥ created_at` | coerce, then compare | HIGH: lateness against an impossible promise is meaningless |
| Event chronology is valid | `created ≤ pickup ≤ delivered` | pairwise comparison | MEDIUM: stage metrics go negative |
| "Late" has an agreed definition | one owner, one formula | `client_metric_definitions.json` | HIGH: every team publishes a different number |

In [3]:
print("Rows:", len(orders))
print("Unique orders:", orders["order_id"].nunique())
print(orders["final_status"].value_counts(dropna=False))

# TODO:
# - establish business grain
# - check delivered orders with missing completion time
# - test promised_eta >= created_at
# - test pickup_at <= actual_delivery_at
# - mark which assumptions need stakeholder clarification

Rows: 1603
Unique orders: 1600
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64


In [4]:
customers = pd.read_sql("SELECT customer_id FROM customers", con)
a = orders.drop_duplicates("order_id", keep="first").copy()
a["final_status_norm"] = a.final_status.str.strip().str.lower()
for c in ["created_at", "promised_eta", "pickup_at", "actual_delivery_at"]:
    a[c] = pd.to_datetime(a[c], format="mixed", errors="coerce")
checks = pd.Series({
    "conflicting duplicate orders": orders[orders.order_id.duplicated(keep=False)].groupby("order_id").traffic_bucket.nunique().gt(1).sum(),
    "delivered without completion time": ((a.final_status_norm == "delivered") & a.actual_delivery_at.isna()).sum(),
    "promised_eta < created_at": (a.promised_eta < a.created_at).sum(),
    "actual_delivery_at < pickup_at": (a.actual_delivery_at < a.pickup_at).sum(),
    "pickup_at < created_at": (a.pickup_at < a.created_at).sum(),
    "null restaurant_id / driver_id": a.restaurant_id.isna().sum() + a.driver_id.isna().sum(),
}, name="violations")
display(checks)
bad = a[(a.promised_eta < a.created_at) | (a.actual_delivery_at < a.pickup_at)]
bad = bad.assign(promise_minus_created=(bad.promised_eta - bad.created_at).dt.total_seconds() / 60, delivered_minus_pickup=(bad.actual_delivery_at - bad.pickup_at).dt.total_seconds() / 60)
display(bad[["order_id", "created_at", "promised_eta", "pickup_at", "actual_delivery_at", "promise_minus_created", "delivered_minus_pickup"]])

conflicting duplicate orders          3
delivered without completion time    37
promised_eta < created_at             4
actual_delivery_at < pickup_at        5
pickup_at < created_at                0
null restaurant_id / driver_id        6
Name: violations, dtype: int64

,order_id,created_at,promised_eta,pickup_at,actual_delivery_at,promise_minus_created,delivered_minus_pickup
50,O00051,2026-08-01 15:09:00,2026-08-01 16:28:30.928528,2026-08-01 16:24:51.495665,2026-08-01 16:19:51.495665,79.515475,-5.000000
51,O00052,2026-08-24 17:07:00,2026-08-24 18:26:48.000000,2026-08-24 18:16:16.946011,2026-08-24 18:11:16.946011,79.800000,-5.000000
52,O00053,2026-08-13 17:06:00,2026-08-13 17:58:50.809718,2026-08-13 18:25:53.797613,2026-08-13 18:20:53.797613,52.846829,-5.000000
53,O00054,2026-08-28 19:56:00,2026-08-28 21:15:48.000000,2026-08-28 21:25:10.582862,2026-08-28 21:20:10.582862,79.800000,-5.000000
54,O00055,2026-08-16 12:53:00,2026-08-16 14:05:48.000000,2026-08-16 14:08:05.645770,2026-08-16 14:03:05.645770,72.800000,-5.000000
100,O00101,2026-08-24 20:59:00,2026-08-24 20:49:00.000000,2026-08-24 21:25:03.983984,2026-08-24 22:14:30.685278,-10.000000,49.445022
101,O00102,2026-08-11 16:41:00,2026-08-11 16:31:00.000000,2026-08-11 17:05:05.809584,2026-08-11 17:57:17.661772,-10.000000,52.197536
102,O00103,2026-08-22 20:11:00,2026-08-22 20:01:00.000000,2026-08-22 20:39:45.064627,2026-08-22 21:10:17.342305,-10.000000,30.537961
103,O00104,2026-08-09 22:19:00,2026-08-09 22:09:00.000000,2026-08-09 22:47:33.934776,2026-08-09 23:36:20.740161,-10.000000,48.780090


> ✅ **Answer**

**Questions that need a stakeholder, because the data cannot settle them:**
- **Data Team:** why do three orders exist twice with different traffic buckets? Are two systems writing to the table?
- **Fleet Ops:** may the driver-app `delivered` event back-fill the 37 missing delivery times?
- **ETA service:** every broken promise is off by exactly −10 minutes, and every early delivery by exactly −5 minutes. That looks like a writer bug, not noise.

# Challenge 2 — Stakeholders disagree on “late”


Read `client_metric_definitions.json`.

Calculate the metric under at least three definitions:
- any delay > 0 minutes,
- delay > 10 minutes,
- historical-style delivered/non-null population.

| Definition | Late rate | Business meaning |
|---|---:|---|

Then answer:

> **Which one should leadership publish, and who must own that decision?**

In [5]:
analysis = orders.drop_duplicates("order_id", keep="first").copy()

for c in ["promised_eta", "actual_delivery_at"]:
    analysis[c] = pd.to_datetime(analysis[c], format="mixed", errors="coerce")

analysis["delay_min"] = (
    analysis["actual_delivery_at"] - analysis["promised_eta"]
).dt.total_seconds() / 60

# TODO: calculate the late rate using different definitions.

In [6]:
a["delay_min"] = (a.actual_delivery_at - a.promised_eta).dt.total_seconds() / 60
hist = a[(a.final_status_norm == "delivered") & a.actual_delivery_at.notna()]
validated = hist[(hist.promised_eta >= hist.created_at) & ~(hist.actual_delivery_at < hist.pickup_at)]
rows = [("VP Operations: any delay > 0 (historical population)", (hist.delay_min > 0).sum(), len(hist), "any missed promise counts"),
        ("Support Lead: delay > 10 min", (hist.delay_min > 10).sum(), len(hist), "'meaningfully late'"),
        ("Data Team: delivered with non-null actual time", (hist.delay_min > 0).sum(), len(hist), "the dashboard population"),
        ("Finance: cancelled / refunded excluded", (hist.delay_min > 0).sum(), len(hist), "already true in every row above"),
        ("Pipeline: validated population (chronology rules)", (validated.delay_min > 0).sum(), len(validated), "drops 9 contradictory orders")]
t = pd.DataFrame(rows, columns=["definition", "late", "population", "business meaning"])
t.insert(1, "late_rate_%", (100 * t.late / t.population).round(2))
display(t)

,definition,late_rate_%,late,population,business meaning
0,VP Operations: any delay > 0 (historical population),56.39,843,1495,any missed promise counts
1,Support Lead: delay > 10 min,23.34,349,1495,'meaningfully late'
2,Data Team: delivered with non-null actual time,56.39,843,1495,the dashboard population
3,Finance: cancelled / refunded excluded,56.39,843,1495,already true in every row above
4,Pipeline: validated population (chronology rules),56.33,837,1486,drops 9 contradictory orders


> ✅ **Answer**

**Publish:** the VP Operations definition on the validated population, **56.3 %**, with the >10-minute rate (≈ 23 %) shown alongside as a severity measure.

**Owner:** VP Operations must sign the definition. Until then, the pipeline's gate marks KPI ownership as **UNKNOWN**, and every output prints the definition next to the number.

# Challenge 3 — Validate categories without cleaning by instinct


Inspect:
- `final_status`
- `traffic_bucket`
- restaurant `status`
- support-ticket `category`

For each:
1. list observed values,
2. identify representation differences,
3. decide what is safe to normalize,
4. identify what requires owner confirmation.

Remember:

> `"Delivered"` vs `"delivered"` is likely representation.  
> `"handoff"` vs `"handed_off"` may be semantics.

In [7]:
for col in ["final_status", "traffic_bucket"]:
    print("\n", col)
    print(orders[col].value_counts(dropna=False))

print("\nRestaurant status")
print(restaurant_status["status"].value_counts(dropna=False))

print("\nSupport category")
print(tickets["category"].value_counts(dropna=False))


 final_status
final_status
delivered    1530
cancelled      68
Delivered       5
Name: count, dtype: int64

 traffic_bucket
traffic_bucket
medium    635
high      448
low       381
severe    136
HIGH        3
Name: count, dtype: int64

Restaurant status
status
preparing     171
handed_off    168
ready         157
ready           2
READY           1
Ready           1
handoff         1
unknown         1
Name: count, dtype: int64

Support category
category
late_delivery        38
eta_changed          37
restaurant_delay     37
ready_but_waiting    33
status_mismatch      29
driver_not_moving    25
Late Delivery         1
late_delivery         1
ETA issue             1
Name: count, dtype: int64


In [8]:
def inspect(name, s):
    norm = s.astype(str).str.strip().str.lower().str.replace(r"[\s-]+", "_", regex=True)
    print(f"--- {name}: raw values {sorted(map(repr, s.dropna().unique()))}")
    display(pd.DataFrame({"raw": s.value_counts()}).join(pd.DataFrame({"after_representation_fix": norm.value_counts()}), how="outer").fillna(0).astype(int))
inspect("final_status", orders.final_status); inspect("traffic_bucket", orders.traffic_bucket)
inspect("restaurant status", restaurant_status.status); inspect("ticket category", tickets.category)

--- final_status: raw values ["'Delivered'", "'cancelled'", "'delivered'"]


,raw,after_representation_fix
final_status,,
Delivered,5,0
cancelled,68,68
delivered,1530,1535


--- traffic_bucket: raw values ["'HIGH'", "'high'", "'low'", "'medium'", "'severe'"]

,raw,after_representation_fix
traffic_bucket,,
HIGH,3,0
high,448,451
low,381,381
medium,635,635
severe,136,136


--- restaurant status: raw values ["'READY'", "'Ready'", "'handed_off'", "'handoff'", "'preparing'", "'ready '", "'ready'", "'unknown'"]


,raw,after_representation_fix
status,,
READY,1,0
Ready,1,0
handed_off,168,168
handoff,1,1
preparing,171,171
ready,157,161
ready,2,0
unknown,1,1


--- ticket category: raw values ["'ETA issue'", "'Late Delivery'", "'driver_not_moving'", "'eta_changed'", "'late_delivery '", "'late_delivery'", "'ready_but_waiting'", "'restaurant_delay'", "'status_mismatch'"]


,raw,after_representation_fix
category,,
ETA issue,1,0
Late Delivery,1,0
driver_not_moving,25,25
eta_changed,37,37
eta_issue,0,1
late_delivery,38,40
late_delivery,1,0
ready_but_waiting,33,33
restaurant_delay,37,37


> ✅ **Answer**

| Column | Safe to normalise (spelling) | Needs the owner (meaning) |
|---|---|---|
| `final_status` | `Delivered` → `delivered` | — |
| `traffic_bucket` | `HIGH` → `high` | a new value such as `gridlock` would be a new bucket, not `severe` |
| restaurant `status` | `READY`, `Ready`, `ready ` → `ready` | `handoff` vs `handed_off` (**Restaurant Ops**); `unknown` is a real value |
| ticket `category` | `Late Delivery`, `late_delivery ` → `late_delivery` | `ETA issue` vs `eta_changed` (**Support Lead**) |

The pipeline applies exactly this split. Representation fixes are logged in `output/data_quality_report.md`. Semantic look-alikes are reported and never merged, and the hidden-data tests check that.

# Challenge 4 — Cross-source integrity


Validate mappings:
- `orders.restaurant_id` → restaurants
- `orders.driver_id` → drivers
- `tickets.order_id` → orders
- `restaurant_status.order_id` → orders

Produce:

| Relationship | Coverage % | Status | Risk |
|---|---:|---|---|

Then discuss:

> If 1% is unmapped, is that acceptable?

It depends on which business decision uses those records.

In [9]:
# Example:
# orders["restaurant_id"].dropna().isin(restaurants["restaurant_id"]).mean()

# TODO: calculate mapping coverage for all relationships.

In [10]:
rel = [("orders.restaurant_id → restaurants", a.restaurant_id.isin(restaurants.restaurant_id).mean(), "3 NULL restaurant_id"),
       ("orders.driver_id → drivers", a.driver_id.isin(drivers.driver_id).mean(), "3 NULL driver_id; 95 stale after reassignment"),
       ("orders.customer_id → customers", a.customer_id.isin(customers.customer_id).mean(), ""),
       ("tickets.order_id → orders", tickets.order_id.isin(a.order_id).mean(), "3 tickets without order_id"),
       ("restaurant_status.order_id → orders", restaurant_status.order_id.isin(a.order_id).mean(), "only 500 of 1600 orders have a status row")]
cov = pd.DataFrame(rel, columns=["relationship", "coverage", "note"])
cov["coverage_%"] = (100 * cov.coverage).round(2)
cov["status"] = cov["coverage_%"].map(lambda v: "PASS" if v == 100 else ("WARN" if v >= 95 else "FAIL"))
display(cov[["relationship", "coverage_%", "status", "note"]])
# the check the class did not ask for: does orders.driver_id match Dispatch's FINAL driver?
pages = sorted(max((ROOT / "data" / "raw").glob("run_*/api"), key=lambda p: p.stat().st_mtime).glob("dispatch_page_*.json"))
dispatch = pd.DataFrame([r for p in pages for r in json.loads(p.read_text())["payload"]["data"]])
m = a.merge(dispatch[["order_id", "driver_id", "original_driver_id"]], on="order_id", suffixes=("", "_dispatch"))
print("orders.driver_id == Dispatch final driver:", (m.driver_id == m.driver_id_dispatch).sum(), "| == Dispatch ORIGINAL driver:", (m.driver_id == m.original_driver_id).sum(), "of", len(m))

,relationship,coverage_%,status,note
0,orders.restaurant_id → restaurants,99.81,WARN,3 NULL restaurant_id
1,orders.driver_id → drivers,99.81,WARN,3 NULL driver_id; 95 stale after reassignment
2,orders.customer_id → customers,100.00,PASS,
3,tickets.order_id → orders,98.51,WARN,3 tickets without order_id
4,restaurant_status.order_id → orders,100.00,PASS,only 500 of 1600 orders have a status row


orders.driver_id == Dispatch final driver: 1502 | == Dispatch ORIGINAL driver: 1597 of 1600


> ✅ **Answer**

**Is 1 % unmapped acceptable?** It depends on the decision.
- **For the late-delivery rate, yes.** The KPI needs only the order's own timestamps.
- **For restaurant or driver scorecards, no.** Those need every order mapped.

The bigger finding is a mapping that *looks* complete but is wrong: `orders.driver_id` equals Dispatch's **original** driver for 1597 orders. The orders table is never updated after a reassignment, so a driver scorecard built on it would blame the wrong person in 95 cases.

# Challenge 5 — Freshness is an SLA question


Use `restaurant_status.csv`.

Determine whether status updates are fresh enough for:
- weekly analytics,
- live customer ETA,
- restaurant accountability.

The same record may be acceptable for one use case and unsafe for another.

**Hint:** join status records to order lifecycle timestamps and inspect timing.

In [11]:
# TODO:
# Merge restaurant_status with orders on order_id
# Parse timestamps
# Compare last_updated_at with order lifecycle timestamps

In [12]:
rs = restaurant_status.drop_duplicates().assign(status_norm=lambda d: d.status.str.strip().str.lower(),
                                                  last_updated_at=lambda d: pd.to_datetime(d.last_updated_at, format="mixed"))
m = rs.merge(a[["order_id", "created_at", "pickup_at"]], on="order_id", how="left")
m["lag_vs_pickup_min"] = (m.last_updated_at - m.pickup_at).dt.total_seconds() / 60
print("orders covered:", m.order_id.nunique(), "of", len(a), "| precision: minutes only")
display(m.groupby("status_norm").lag_vs_pickup_min.describe().round(1))
print("ready/handed_off written > 15 min AFTER pickup:", (m.status_norm.isin(["ready", "handed_off"]) & (m.lag_vs_pickup_min > 15)).sum(),
      "| written BEFORE the order existed:", (m.last_updated_at < m.created_at).sum())

orders covered: 500 of 1600 | precision: minutes only


,count,mean,std,min,25%,50%,75%,max
status_norm,,,,,,,,
handed_off,168.0,-8.9,30.5,-368.5,-15.5,-6.4,3.0,17.6
handoff,1.0,8.1,NaN,8.1,8.1,8.1,8.1,8.1
preparing,170.0,-9.9,32.1,-380.4,-16.4,-7.3,2.0,23.7
ready,160.0,-12.9,51.9,-385.8,-17.2,-5.2,5.3,25.4
unknown,1.0,-3.2,NaN,-3.2,-3.2,-3.2,-3.2,-3.2


ready/handed_off written > 15 min AFTER pickup: 8 | written BEFORE the order existed: 5


> ✅ **Answer**

| Use case | Fresh enough? | Why |
|---|---|---|
| Weekly analytics | **WARN, usable** | covers 31 % of orders at minute precision; fine for aggregates if the coverage bias is stated |
| Live customer ETA | **FAIL** | some `ready` updates arrive after pickup, and some before the order exists |
| Restaurant accountability | **FAIL** | 38 % of restaurants update manually, so a late `ready` may be a late *update*, not a late *kitchen* |

# Challenge 6 — Build the validation gate


Summarize the investigation:

| Check | Status | Evidence | Action |
|---|---|---|---|
| Business grain |  |  |  |
| Timestamp chronology |  |  |  |
| KPI definition |  |  |  |
| Category semantics |  |  |  |
| Cross-source mapping |  |  |  |
| Freshness |  |  |  |

Use only:
**PASS / WARN / FAIL / UNKNOWN**

Finally answer:

> **Should leadership publish “Late Delivery Rate = 56%” today?**

What exactly must happen before the metric becomes publishable?

In [13]:
validation_report = {
    "business_grain": None,
    "timestamp_chronology": None,
    "kpi_definition": None,
    "category_semantics": None,
    "cross_source_mapping": None,
    "freshness": None,
    "publish_56_percent": None
}
print(validation_report)

{'business_grain': None, 'timestamp_chronology': None, 'kpi_definition': None, 'category_semantics': None, 'cross_source_mapping': None, 'freshness': None, 'publish_56_percent': None}


In [14]:
validation_report = {
    "business_grain":       ("WARN",    "1603 rows / 1600 orders; 3 conflicting duplicates kept-first and quarantined"),
    "timestamp_chronology": ("WARN",    "4 promises before creation (−10 min), 5 deliveries before pickup (−5 min), 37 missing delivery times"),
    "kpi_definition":       ("UNKNOWN", "4 stakeholder definitions, no documented owner"),
    "category_semantics":   ("WARN",    "spelling fixed; 'handoff' and 'ETA issue' kept separate for their owners"),
    "cross_source_mapping": ("WARN",    "99.8 % restaurant/driver, 98.5 % tickets; orders.driver_id stale after reassignment"),
    "freshness":            ("WARN",    "restaurant feed: 31 % coverage, minute precision, stale and pre-creation updates"),
    "weather_label":        ("WARN",    "beyond the class: client weather label vs observed Open-Meteo rainfall, kappa ≈ 0"),
    "publish_56_percent":   ("PUBLISH WITH CAVEATS", "56.3 % of 1486 validated deliveries (56.4 % on the dashboard definition)"),
}
display(pd.DataFrame(validation_report, index=["status", "evidence"]).T)
gate_file = ROOT / "output" / "validation_gate.json"
if gate_file.exists():
    g = json.loads(gate_file.read_text(encoding="utf-8"))
    print("pipeline gate of the last published run:", g["overall_status"], "->", g["publish_decision"])

,status,evidence
business_grain,WARN,1603 rows / 1600 orders; 3 conflicting duplicates kept-first and quarantined
timestamp_chronology,WARN,"4 promises before creation (−10 min), 5 deliveries before pickup (−5 min), 37 missing ..."
kpi_definition,UNKNOWN,"4 stakeholder definitions, no documented owner"
category_semantics,WARN,spelling fixed; 'handoff' and 'ETA issue' kept separate for their owners
cross_source_mapping,WARN,"99.8 % restaurant/driver, 98.5 % tickets; orders.driver_id stale after reassignment"
freshness,WARN,"restaurant feed: 31 % coverage, minute precision, stale and pre-creation updates"
weather_label,WARN,"beyond the class: client weather label vs observed Open-Meteo rainfall, kappa ≈ 0"
publish_56_percent,PUBLISH WITH CAVEATS,56.3 % of 1486 validated deliveries (56.4 % on the dashboard definition)


pipeline gate of the last published run: WARN -> PUBLISH WITH CAVEATS: late delivery rate = 56.33% of 1486 validated deliveries (historical definition: 56.39%). State the definition, the excluded orders and the unresolved owner questions next to the number.


> ✅ **Answer**

**Should leadership publish "Late Delivery Rate = 56 %" today?** Yes, **with caveats**, never as a bare number:

> "56.3 % of validated August deliveries (837 of 1486) missed the promised ETA. 37 deliveries have an unknown outcome and 9 were excluded for contradictory timestamps. Definition owned by VP Operations (pending)."

**Before it can be published without caveats:**
1. VP Operations signs the definition.
2. Fleet Ops fixes or approves the back-fill of the missing delivery times.
3. The ETA service explains the fixed −10 / −5 minute offsets.

# Final takeaway

The job was not to make the data look clean.

The job was to decide:

> **Is this data safe enough for this decision, and what remains unresolved?**